# 制造业AI智慧检测物联网平台 - Colab实验笔记本

这个笔记本用于在Google Colab上进行数据分析和机器学习实验。

## 功能
- 加载轮检测数据
- 数据可视化
- 质量检测分析
- 机器学习模型实验

In [ ]:
# 安装必要的库
!pip install pandas numpy matplotlib seaborn plotly scikit-learn

In [ ]:
# 导入库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import json

## 1. 数据加载

In [ ]:
# 从GitHub加载数据文件
data_url = "https://raw.githubusercontent.com/XSX-SXS/wheel-detection-platform/main/data.json"

# 或者使用本地数据（如果已上传）
# data_url = "data.json"

try:
    df = pd.read_json(data_url)
    print(f"数据加载成功！共有 {len(df)} 条记录")
    print(df.head())
except Exception as e:
    print(f"从GitHub加载数据失败: {e}")
    print("请手动上传 data.json 文件到Colab")
    # 创建示例数据
    sample_data = [
        {"id": "202503110001", "diameter": 650, "average_bolt": 48, "center": 80, "pcd": 280, "type": "合格"},
        {"id": "202503110002", "diameter": 650, "average_bolt": 48, "center": 80, "pcd": 280, "type": "合格"},
        {"id": "202503110003", "diameter": 650, "average_bolt": 48, "center": 80, "pcd": 280, "type": "不合格"},
        {"id": "202503110004", "diameter": 650, "average_bolt": 48, "center": 80, "pcd": 280, "type": "合格"},
        {"id": "202503110005", "diameter": 650, "average_bolt": 48, "center": 80, "pcd": 280, "type": "合格"}
    ]
    df = pd.DataFrame(sample_data)
    print("使用示例数据")
    print(df.head())

## 2. 数据探索

In [ ]:
# 数据基本信息
print("数据形状:", df.shape)
print("\n数据类型:")
print(df.dtypes)
print("\n基本统计信息:")
print(df.describe())
print("\n质量分布:")
print(df['type'].value_counts())

## 3. 数据可视化

In [ ]:
# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 质量分布饼图
fig, ax = plt.subplots(figsize=(8, 6))
quality_counts = df['type'].value_counts()
colors = ['#2ecc71', '#e74c3c']  # 绿色表示合格，红色表示不合格
ax.pie(quality_counts.values, labels=quality_counts.index, autopct='%1.1f%%', 
       colors=colors, startangle=90)
ax.set_title('产品质量分布', fontsize=16, fontweight='bold')
plt.show()

# 显示具体数量
print("质量统计:")
for quality, count in quality_counts.items():
    percentage = (count / len(df)) * 100
    print(f"{quality}: {count} 件 ({percentage:.1f}%)")

In [ ]:
# 参数分布直方图
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 直径分布
axes[0, 0].hist(df['diameter'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].set_title('直径分布')
axes[0, 0].set_xlabel('直径 (mm)')
axes[0, 0].set_ylabel('频次')

# 螺栓平均分布
axes[0, 1].hist(df['average_bolt'], bins=20, alpha=0.7, color='lightgreen', edgecolor='black')
axes[0, 1].set_title('螺栓平均分布')
axes[0, 1].set_xlabel('螺栓平均值')
axes[0, 1].set_ylabel('频次')

# 中心孔分布
axes[1, 0].hist(df['center'], bins=20, alpha=0.7, color='orange', edgecolor='black')
axes[1, 0].set_title('中心孔分布')
axes[1, 0].set_xlabel('中心孔 (mm)')
axes[1, 0].set_ylabel('频次')

# PCD分布
axes[1, 1].hist(df['pcd'], bins=20, alpha=0.7, color='pink', edgecolor='black')
axes[1, 1].set_title('PCD分布')
axes[1, 1].set_xlabel('PCD (mm)')
axes[1, 1].set_ylabel('频次')

plt.tight_layout()
plt.show()

In [ ]:
# 参数相关性热力图
numeric_cols = ['diameter', 'average_bolt', 'center', 'pcd']
correlation_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5)
plt.title('参数相关性热力图')
plt.show()

## 4. 交互式可视化

In [ ]:
# 创建交互式散点图
fig = px.scatter(df, x='diameter', y='pcd', color='type', 
               size='average_bolt', hover_data=['id', 'center'],
               title='直径 vs PCD 散点图 (按质量分类)')
fig.show()

In [ ]:
# 3D散点图
fig = go.Figure(data=go.Scatter3d(
    x=df['diameter'],
    y=df['average_bolt'],
    z=df['pcd'],
    mode='markers',
    marker=dict(
        size=6,
        color=['red' if t == '不合格' else 'green' for t in df['type']],
        opacity=0.8
    ),
    text=df['id'],
    hovertemplate='<b>%{text}</b><br>直径: %{x}<br>螺栓: %{y}<br>PCD: %{z}<extra></extra>'
))

fig.update_layout(
    title='3D参数分布图',
    scene=dict(
        xaxis_title='直径 (mm)',
        yaxis_title='螺栓平均值',
        zaxis_title='PCD (mm)'
    )
)
fig.show()

## 5. 机器学习实验

In [ ]:
# 准备训练数据
X = df[['diameter', 'average_bolt', 'center', 'pcd']]
y = df['type']

# 将分类标签转换为数值
y_numeric = (y == '合格').astype(int)

# 分割训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y_numeric, test_size=0.2, random_state=42, stratify=y_numeric
)

print(f"训练集大小: {len(X_train)}")
print(f"测试集大小: {len(X_test)}")
print(f"训练集合格率: {y_train.mean():.2f}")
print(f"测试集合格率: {y_test.mean():.2f}")

In [ ]:
# 训练随机森林模型
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 预测
y_pred = rf_model.predict(X_test)

# 评估模型
print("分类报告:")
print(classification_report(y_test, y_pred, target_names=['不合格', '合格']))

# 特征重要性
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n特征重要性:")
print(feature_importance)

In [ ]:
# 特征重要性可视化
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('重要性')
plt.title('特征重要性分析')
plt.gca().invert_yaxis()
plt.show()

# 混淆矩阵
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['不合格', '合格'], yticklabels=['不合格', '合格'])
plt.xlabel('预测标签')
plt.ylabel('真实标签')
plt.title('混淆矩阵')
plt.show()

## 6. 异常检测实验

In [ ]:
# 使用统计方法检测异常值
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return (data[column] < lower_bound) | (data[column] > upper_bound)

# 检测每个参数的异常值
outlier_results = {}
for col in ['diameter', 'average_bolt', 'center', 'pcd']:
    outliers = detect_outliers_iqr(df, col)
    outlier_results[col] = outliers.sum()
    print(f"{col}: 发现 {outliers.sum()} 个异常值")
    if outliers.sum() > 0:
        print(f"  异常值范围: {df.loc[outliers, col].min()} - {df.loc[outliers, col].max()}")
        print(f"  正常值范围: {df.loc[~outliers, col].min()} - {df.loc[~outliers, col].max()}")
    print()

In [ ]:
# 异常值可视化
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for i, col in enumerate(['diameter', 'average_bolt', 'center', 'pcd']):
    outliers = detect_outliers_iqr(df, col)
    
    # 箱线图
    axes[i].boxplot(df[col], vert=True)
    axes[i].set_title(f'{col} 箱线图')
    axes[i].set_ylabel(col)
    
    # 标记异常值
    if outliers.sum() > 0:
        outlier_values = df.loc[outliers, col]
        axes[i].scatter([1] * len(outlier_values), outlier_values, 
                       color='red', s=50, alpha=0.7, label='异常值')
        axes[i].legend()

plt.tight_layout()
plt.show()

## 7. 质量预测实验

In [ ]:
# 创建新的特征
df_features = df.copy()
df_features['diameter_center_ratio'] = df_features['diameter'] / df_features['center']
df_features['pcd_diameter_ratio'] = df_features['pcd'] / df_features['diameter']
df_features['bolt_density'] = df_features['average_bolt'] / df_features['diameter']

# 添加异常值标记
for col in ['diameter', 'average_bolt', 'center', 'pcd']:
    df_features[f'{col}_outlier'] = detect_outliers_iqr(df_features, col).astype(int)

print("新特征:")
print(df_features[['diameter_center_ratio', 'pcd_diameter_ratio', 'bolt_density']].describe())

In [ ]:
# 使用扩展特征训练模型
feature_cols = ['diameter', 'average_bolt', 'center', 'pcd', 
                'diameter_center_ratio', 'pcd_diameter_ratio', 'bolt_density',
                'diameter_outlier', 'average_bolt_outlier', 'center_outlier', 'pcd_outlier']

X_extended = df_features[feature_cols]
y_extended = (df_features['type'] == '合格').astype(int)

# 分割数据
X_train_ext, X_test_ext, y_train_ext, y_test_ext = train_test_split(
    X_extended, y_extended, test_size=0.2, random_state=42, stratify=y_extended
)

# 训练模型
rf_extended = RandomForestClassifier(n_estimators=200, random_state=42)
rf_extended.fit(X_train_ext, y_train_ext)

# 评估
y_pred_ext = rf_extended.predict(X_test_ext)
print("扩展特征模型性能:")
print(classification_report(y_test_ext, y_pred_ext, target_names=['不合格', '合格']))

# 特征重要性对比
importance_extended = pd.DataFrame({
    'feature': X_extended.columns,
    'importance': rf_extended.feature_importances_
}).sort_values('importance', ascending=False)

print("\n扩展特征重要性:")
print(importance_extended.head(10))

## 8. 结果保存与导出

In [ ]:
# 保存分析结果
results = {
    '数据概览': {
        '总记录数': len(df),
        '合格率': (df['type'] == '合格').mean(),
        '不合格率': (df['type'] == '不合格').mean()
    },
    '模型性能': {
        '基础模型准确率': (y_pred == y_test).mean(),
        '扩展模型准确率': (y_pred_ext == y_test_ext).mean()
    },
    '重要发现': {
        '最重要特征': importance_extended.iloc[0]['feature'],
        '异常值总数': sum(outlier_results.values())
    }
}

print("实验结果总结:")
for category, data in results.items():
    print(f"\n{category}:")
    for key, value in data.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.3f}")
        else:
            print(f"  {key}: {value}")

# 保存到文件
with open('experiment_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("\n结果已保存到 experiment_results.json")

## 9. 下一步实验建议

基于当前分析，您可以尝试:

1. **深度学习模型**: 尝试使用神经网络进行质量预测
2. **时间序列分析**: 如果有时间数据，分析质量趋势
3. **聚类分析**: 发现数据中的隐藏模式
4. **异常检测算法**: 尝试Isolation Forest或One-Class SVM
5. **特征工程**: 创建更多有意义的特征

这个笔记本提供了完整的实验框架，您可以根据需要修改和扩展！